# Traffic Data Analysis

exploring the smart traffic dataset from kaggle. mainly want to look at:
- when does traffic actually peak during the day
- does giving more green time actually reduce how long vehicles wait

have seen this stuff in real systems at work so curious how the data holds up

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

### load data

In [ ]:
df = pd.read_sql('select * from traffic_observations', engine, parse_dates=['timestamp'])
df.shape

In [ ]:
df.head(10)

In [ ]:
# checking nulls before anything else
df.isnull().sum()

In [ ]:
df.describe()

### feature engineering

need hour and day columns for the time-based stuff

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_name'] = df['timestamp'].dt.day_name()
df['day_num'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['day_num'].isin([5, 6])

df[['timestamp', 'hour', 'day_name', 'is_weekend']].head()

In [ ]:
# rough time of day buckets
def tod(h):
    if 6 <= h < 10: return 'morning peak'
    if 10 <= h < 17: return 'midday'
    if 17 <= h < 21: return 'evening peak'
    return 'night'

df['time_of_day'] = df['hour'].apply(tod)
df['time_of_day'].value_counts()

---
## part 1 — peak hours

In [ ]:
hourly = df.groupby('hour')['vehicle_count'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hourly['hour'], hourly['vehicle_count'], color='steelblue', width=0.7)
ax.set_xlabel('hour of day')
ax.set_ylabel('avg vehicle count')
ax.set_title('average traffic by hour')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig('../visuals/peak_hour_bar.png')
plt.show()

In [ ]:
# top 5 peak hours
hourly.sort_values('vehicle_count', ascending=False).head(5)

makes sense — morning and evening double peak. pretty standard for urban traffic

In [ ]:
# weekday vs weekend — curious if weekend pattern is different
wd = df.groupby(['hour', 'is_weekend'])['vehicle_count'].mean().reset_index()
wd['label'] = wd['is_weekend'].map({False: 'weekday', True: 'weekend'})

fig, ax = plt.subplots(figsize=(12, 4))
for label, grp in wd.groupby('label'):
    ax.plot(grp['hour'], grp['vehicle_count'], marker='o', label=label)
ax.set_xlabel('hour')
ax.set_ylabel('avg vehicle count')
ax.set_title('weekday vs weekend traffic')
ax.legend()
plt.tight_layout()
plt.savefig('../visuals/weekday_vs_weekend.png')
plt.show()

In [ ]:
# heatmap — hour vs day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = df.groupby(['day_name', 'hour'])['vehicle_count'].mean().unstack()
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.2, ax=ax)
ax.set_title('traffic by hour and day')
ax.set_xlabel('hour')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../visuals/heatmap.png')
plt.show()

---
## part 2 — signal timing vs wait time

the main question here: if you give more green time, does wait time actually go down? intuition says yes but it might depend on volume

In [ ]:
sig = df.dropna(subset=['green_duration_sec', 'wait_time_sec']).copy()
print(sig.shape)
sig[['green_duration_sec', 'wait_time_sec']].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    sig['green_duration_sec'],
    sig['wait_time_sec'],
    alpha=0.25,
    c=sig['vehicle_count'],
    cmap='coolwarm',
    s=12
)
plt.colorbar(scatter, ax=ax, label='vehicle count')

# trend line
z = np.polyfit(sig['green_duration_sec'], sig['wait_time_sec'], 1)
p = np.poly1d(z)
x_range = np.linspace(sig['green_duration_sec'].min(), sig['green_duration_sec'].max(), 200)
ax.plot(x_range, p(x_range), 'k-', linewidth=1.5, label='trend')

ax.set_xlabel('green duration (sec)')
ax.set_ylabel('wait time (sec)')
ax.set_title('green duration vs wait time (color = vehicle count)')
ax.legend()
plt.tight_layout()
plt.savefig('../visuals/green_vs_wait.png')
plt.show()

print('correlation:', sig['green_duration_sec'].corr(sig['wait_time_sec']).round(3))

interesting — update this once you see the correlation value. if it's positive that means longer green isn't helping (probably because those intersections are busier to begin with)

In [ ]:
# bucket green durations and compare avg wait
sig['green_bucket'] = pd.cut(
    sig['green_duration_sec'],
    bins=[0, 30, 60, 90, 999],
    labels=['<30s', '30-60s', '60-90s', '>90s']
)

bucket_agg = sig.groupby('green_bucket', observed=True).agg(
    avg_wait=('wait_time_sec', 'mean'),
    avg_volume=('vehicle_count', 'mean'),
    n=('id', 'count')
).reset_index()

bucket_agg

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.bar(bucket_agg['green_bucket'], bucket_agg['avg_wait'], color='steelblue', alpha=0.7)
ax2.plot(bucket_agg['green_bucket'], bucket_agg['avg_volume'], color='tomato', marker='o', linewidth=2)

ax1.set_ylabel('avg wait time (sec)', color='steelblue')
ax2.set_ylabel('avg vehicle count', color='tomato')
ax1.set_xlabel('green duration bucket')
ax1.set_title('wait time vs volume by green duration')
plt.tight_layout()
plt.savefig('../visuals/bucket_analysis.png')
plt.show()

In [ ]:
# wait time by signal phase
phase = df.groupby('signal_phase')[['wait_time_sec', 'vehicle_count']].mean().dropna()
phase

In [ ]:
colors = {'green': '#2ecc71', 'yellow': '#f1c40f', 'red': '#e74c3c'}
c = [colors.get(p, 'gray') for p in phase.index]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(phase.index, phase['wait_time_sec'], color=c, edgecolor='white')
ax.set_ylabel('avg wait time (sec)')
ax.set_title('wait time by signal phase')
plt.tight_layout()
plt.savefig('../visuals/phase_wait.png')
plt.show()

---
## findings

**peak hours**
- clear double peak pattern — morning (8-9am) and evening (6-7pm) as expected
- weekends have a later, flatter peak — no morning rush
- update with actual top hours after running

**signal timing vs wait time**
- correlation value: [fill in after running]
- the scatter suggests high-volume intersections have both long green time AND long wait — probably because the signal is already compensating for heavy load
- signal phase analysis confirms red phases have the highest wait, yellow is short enough it barely registers